# The Empathetic Fitting Room: Results Analysis

This notebook reads all 11 participants' session data and calculates:

1. How much their heart rate (RR-interval) changed during each condition
2. How much their SAM score changed during each condition
3. Whether the adaptive condition was statistically better than the static condition (Bayesian test)
4. Whether the system stayed under 400ms per heartbeat

**Note:** This notebook requires the raw adaptive session CSVs (`01_adaptive.csv`-`11_adaptive.csv`) and the CSV containing the SAM scores, file links and condition order (`participant_info.csv`) which are not included in the supporting material submission, consistent with the data access terms  agreed with participants (see Reflective Essay, Section 6). This notebook's cells retain their originally executed output below, so results remain visible and verifiable by inspection without requiring the underlying raw data.

In [1]:
import csv
import pingouin as pg

## 1. Participant data

For each participant we store:
- which CSV files hold their heartbeat data
- their SAM stress scores (1 = very calm, 9 = very stressed) at three points: before stress induction, after the adaptive condition, and after the static condition
- their condition order (static or adaptive first)

In [2]:
def load_participants(filename="participant_info.csv"):
    """
    Loads participant info (file links, SAM scores, condition order) from participant_info.csv. 
    This file is not included in the supporting material submission, 
    consistent with the data access terms agreed with participants (see Reflective Essay, Section 6).
    """
    participants = []
    with open(filename, "r") as f:
        reader = csv.DictReader(f) # Allows to refer to columns by name
        for row in reader:
            participants.append({ # Building a dictionary for each row
                "id": row["participant_id"],
                "adaptive_file": row["adaptive_file"],
                "static_file": row["static_file"],
                "sam_baseline": int(row["sam_baseline"]),
                "sam_adaptive": int(row["sam_adaptive"]),
                "sam_static": int(row["sam_static"]),
                "order": row["order"],
            })
    return participants


participants = load_participants()

## 2. Simple helper functions

This section defines foundational functions that are invoked repeatedly throughout the notebook.

In [3]:
def work_out_average(list_of_numbers):
    total = 0
    for number in list_of_numbers:
        total += number
    return total / len(list_of_numbers)


def read_adaptive_csv(filename):
    """
    Reads one adaptive-condition CSV file.
    Returns three lists: RR-interval per heartbeat, PID reaction score per heartbeat, and how long each cycle took to process (ms).
    """
    rr_values = []
    pid_scores = []
    cycle_times = []

    with open(filename, "r") as f:
        reader = csv.DictReader(f) # DictReader looks at the header and allows to look for data by column name
        for row in reader:
            try:
                rr_values.append(float(row["rr_avg"]))
                pid_scores.append(float(row["reaction_score"]))
                cycle_times.append(float(row["cycle_time_ms"]))
            except (ValueError, KeyError): # If system finds a bad value or missing column header it moves to next row
                pass

    return rr_values, pid_scores, cycle_times


def read_static_csv(filename):
    """Reads a static condition CSV file and returns a list of RR-intervals."""
    rr_values = []
    with open(filename, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                rr_values.append(float(row["rr_interval"]))
            except (ValueError, KeyError):
                pass
    return rr_values

## 3. Calculating RR-interval change

For each session we compare the first third of the session against the last third. This tells us whether the person's heart rate calmed down or stayed stressed by the end.

Note: "first third" here is not a true resting baseline: participants were already stressed from the induction task before the session began, so this comparison shows relative change from that already stressed starting point, not a return to true calm.

In [4]:
def work_out_rr_change(rr_list):
    """
    Compares the first third of a session to the last third.
    Positive change = RR interval got longer = heart slowed down = calming.
    Negative change = RR interval got shorter = heart sped up = more stressed.
    """
    total_beats = len(rr_list)
    third_size = max(1, total_beats // 3) # If sensor glitched and list only had 2 heartbeats, dividing by 3 would equal zero; this forces the answer to be at least 1

    first_third = rr_list[:third_size]
    last_third = rr_list[-third_size:]

    first_average = work_out_average(first_third)
    last_average = work_out_average(last_third)

    change = last_average - first_average
    return round(change, 1), round(first_average, 1), round(last_average, 1)

## 4. The Bayesian statistics function

Traditional frequentist approaches can be limited when evaluating small sample sizes (*N* = 11). To provide a stronger evaluation, this section implements a Bayes Factor (*BF10*) analysis. The *BF10* quantifies the strength of evidence by comparing the probability of the data under the alternative hypothesis (that the adaptive condition produces a measurable effect) against the null hypothesis (that there is no difference).

**Interpretation of the Bayes Factor (Jeffreys, 1961; Wetzels *et al.*, 2011)**
- BF10 > 100 → extreme evidence FOR
- BF10 > 30 AND BF10 <= 100 → very strong evidence FOR
- BF10 > 10 AND BF10 <= 30 → strong evidence FOR
- BF10 > 3 AND BF10 <= 10 → moderate evidence FOR
- BF10 > 1 AND BF10 <= 3 → weak or anecdotal evidence FOR
- BF10 > 0.333 AND BF10 <= 1 → inconclusive: no meaningful evidence either way
- BF10 > 0.1 AND BF10 <= 0.333 → moderate evidence AGAINST
- BF10 <= 0.1 → strong evidence AGAINST

In [ ]:
def calculate_bf10(list_x, list_y, direction):
    "Uses Pingouin to calculate the Bayes Factor for paired data. It returns the standard two-sided result." # Two-sided checks if there is a difference between the two conditions in any direction
    if direction == "greater": # If the Adaptive heart rate change is greater, it sets alternative hypothesis value to "greater"
        alt = "greater"
    elif direction == "less":
        alt = "less"
    else: # If no prediction is provided
        alt = "two-sided"

    result = pg.ttest(list_x, list_y, paired=True, alternative=alt) # runs a paired t-test on data using Pingouin
    t_statistic = result['T'].values[0] # Pulling out t-static and degrees of freedom: how many independent data points went into calculation
    df = result['dof'].values[0]

    bf10 = pg.bayesfactor_ttest(t_statistic, nx=len(list_x), paired=True) # Calculated two_sided BF10

    return round(bf10, 3), round(t_statistic, 3), int(df)


def describe_bf10(bf10_value):
    """Turns a BF10 number into a clear interpretation, as per Jeffreys (1961) and Wetzels et al. (2011)."""
    if bf10_value > 100:
        return "extreme evidence FOR the hypothesis"
    elif bf10_value > 30:
        return "very strong evidence FOR the hypothesis"
    elif bf10_value > 10:
        return "strong evidence FOR the hypothesis"
    elif bf10_value > 3:
        return "moderate evidence FOR the hypothesis"
    elif bf10_value > 1:
        return "weak or anecdotal evidence FOR the hypothesis"
    elif bf10_value > 0.333:
        return "inconclusive: no meaningful evidence either way"
    elif bf10_value > 0.1:
        return "moderate evidence AGAINST the hypothesis"
    else:
        return "strong evidence AGAINST the hypothesis"

## 5. Individual results

In [6]:
print("INDIVIDUAL RESULTS ANALYSIS")

all_rr_change_adaptive = []
all_rr_change_static = []
all_sam_change_adaptive = []
all_sam_change_static = []
all_cycle_times = []
participant_labels = []

for p in participants: # Looping through each participant

    rr_adaptive, pid_scores, cycle_times = read_adaptive_csv(p["adaptive_file"]) # Getting the data from the files
    rr_static = read_static_csv(p["static_file"])

    rr_change_a, start_a, end_a = work_out_rr_change(rr_adaptive) # Calculating heart rate changes (saving the start and end scores)
    rr_change_s, start_s, end_s = work_out_rr_change(rr_static)

    sam_change_a = p["sam_adaptive"] - p["sam_baseline"] # Calculating SAM changes by subtracting baseline
    sam_change_s = p["sam_static"] - p["sam_baseline"]

    pid_average = round(work_out_average(pid_scores), 1) # Calculating PID average
    
    # Saving participant's final numbers into lists
    all_rr_change_adaptive.append(rr_change_a)
    all_rr_change_static.append(rr_change_s)
    all_sam_change_adaptive.append(sam_change_a)
    all_sam_change_static.append(sam_change_s)
    all_cycle_times.extend(cycle_times)
    participant_labels.append(p["id"])

    # Printing out a simple report for each individual
    print("Participant: " + p["id"])
    print(f"Adaptive -> RR change: {round(rr_change_a, 1)}ms (from {round(start_a)} to {round(end_a)}) | SAM: {sam_change_a} | PID: {pid_average}")
    print(f"Static -> RR change: {round(rr_change_s, 1)}ms (from {round(start_s)} to {round(end_s)}) | SAM: {sam_change_s}")
    print() # Prints a blank line between participants

INDIVIDUAL RESULTS ANALYSIS
Participant: P01
Adaptive -> RR change: 16.1ms (from 712 to 728) | SAM: -1 | PID: 33.8
Static -> RR change: -16.1ms (from 684 to 668) | SAM: 2

Participant: P02
Adaptive -> RR change: -24.1ms (from 752 to 727) | SAM: 0 | PID: 26.7
Static -> RR change: -7.9ms (from 735 to 727) | SAM: 0

Participant: P03
Adaptive -> RR change: -49.4ms (from 878 to 828) | SAM: -1 | PID: 31.3
Static -> RR change: -50.2ms (from 858 to 808) | SAM: 2

Participant: P04
Adaptive -> RR change: 10.3ms (from 991 to 1001) | SAM: -4 | PID: 26.7
Static -> RR change: -10.4ms (from 894 to 884) | SAM: -1

Participant: P05
Adaptive -> RR change: -13.0ms (from 1009 to 996) | SAM: 0 | PID: 22.1
Static -> RR change: 1.9ms (from 942 to 944) | SAM: 3

Participant: P06
Adaptive -> RR change: 44.3ms (from 852 to 896) | SAM: -1 | PID: -5.0
Static -> RR change: -51.8ms (from 899 to 847) | SAM: 1

Participant: P07
Adaptive -> RR change: -26.1ms (from 649 to 623) | SAM: 1 | PID: -26.6
Static -> RR change

## 6. Group summary

In [7]:
print("GROUP SUMMARY")

# Calculating the overall averages and rounding them
mean_rr_adaptive = round(work_out_average(all_rr_change_adaptive), 1)
mean_rr_static = round(work_out_average(all_rr_change_static), 1)

mean_sam_adaptive = round(work_out_average(all_sam_change_adaptive), 2)
mean_sam_static = round(work_out_average(all_sam_change_static), 2)

total_people = len(participants)

# Printing the averages
print("Number of participants: " + str(total_people))
print()
print(f"Mean RR change (Adaptive): {mean_rr_adaptive}ms")
print(f"Mean RR change (Static): {mean_rr_static}ms")
print()
print(f"Mean SAM change (Adaptive): {mean_sam_adaptive}")
print(f"Mean SAM change (Static): {mean_sam_static}")
print()

# Setting up empty counters starting at zero
number_rr_recovery_adaptive = 0
number_rr_recovery_static = 0
number_sam_better_adaptive = 0
number_sam_better_static = 0
number_adaptive_better_rr = 0


for i in range(total_people):
    
    if all_rr_change_adaptive[i] > 0: # Checking if RR went up for adaptive
        number_rr_recovery_adaptive += 1
        
    if all_rr_change_static[i] > 0: # Checking if RR went up for static
        number_rr_recovery_static += 1

    if all_sam_change_adaptive[i] < 0: # Checking if stress score dropped below 0 for adaptive
        number_sam_better_adaptive += 1
        
    if all_sam_change_static[i] < 0: # Checking if stress score dropped below 0 for static
        number_sam_better_static += 1

    if all_rr_change_adaptive[i] > all_rr_change_static[i]: # Checking if adaptive worked better than static for specific individual
        number_adaptive_better_rr += 1


print("Participants showing RR recovery (positive change):")
print(f"Adaptive: {number_rr_recovery_adaptive} / {total_people}")
print(f"Static: {number_rr_recovery_static} / {total_people}")
print()

print("Participants feeling calmer (negative SAM change):")
print(f"Adaptive: {number_sam_better_adaptive} / {total_people}")
print(f"Static: {number_sam_better_static} / {total_people}")
print()

print(f"Adaptive produced better RR recovery than static: {number_adaptive_better_rr} / {total_people}")

GROUP SUMMARY
Number of participants: 11

Mean RR change (Adaptive): -16.2ms
Mean RR change (Static): -12.8ms

Mean SAM change (Adaptive): -1.18
Mean SAM change (Static): 0.55

Participants showing RR recovery (positive change):
Adaptive: 3 / 11
Static: 2 / 11

Participants feeling calmer (negative SAM change):
Adaptive: 7 / 11
Static: 4 / 11

Adaptive produced better RR recovery than static: 5 / 11


## 7. The Bayesian tests

Test 1 checks the physiological measure (RR-interval).
Test 2 checks the subjective measure (SAM score).
Test 3 checks system latency.

In [8]:
print("BAYESIAN ANALYSIS")
print()

# TEST 1
print("Test 1: RR-interval recovery (physiological)")
print("Hypothesis: adaptive produces greater RR recovery than static")
print()

rr_bf10, rr_t, rr_df = calculate_bf10(all_rr_change_adaptive, all_rr_change_static, direction="greater")
rr_description = describe_bf10(rr_bf10)

print(f"t({rr_df}) = {rr_t}")
print(f"BF10 = {rr_bf10}")
print("Result: " + rr_description)
print()


# TEST 2
print("Test 2: SAM arousal change (subjective)")
print("Hypothesis: adaptive produces greater reduction in SAM than static")
print()

sam_bf10, sam_t, sam_df = calculate_bf10(all_sam_change_adaptive, all_sam_change_static, direction="less")
sam_description = describe_bf10(sam_bf10)

print(f"t({sam_df}) = {sam_t}")
print(f"BF10 = {sam_bf10}")
print("Result: " + sam_description)
print()


# TEST 3
total_cycles = len(all_cycle_times)

print("Test 3: System latency")
print("Checking whether cycle times stayed below 400ms")
print()

# Calculating averages and max speed
mean_cycle = round(work_out_average(all_cycle_times), 1)
max_cycle = round(max(all_cycle_times), 1)

# Couning how many cycles were too slow
cycles_over_400 = 0
for c in all_cycle_times:
    if c > 400:
        cycles_over_400 += 1

# Finding the rest
cycles_under_400 = total_cycles - cycles_over_400
percentage_over = round((cycles_over_400 / total_cycles) * 100, 1)
percentage_under = round((cycles_under_400 / total_cycles) * 100, 1)

print(f"Total cycles logged: {total_cycles}")
print(f"Mean cycle time: {mean_cycle}ms")
print(f"Max cycle time: {max_cycle}ms")
print(f"Cycles over 400ms: {cycles_over_400} ({percentage_over}%)")
print(f"Cycles under 400ms: {cycles_under_400} ({percentage_under}%)")

# Final conclusion based on a 5% threshold
if percentage_over < 5:
    print("Conclusion: Sub-400ms threshold maintained in the vast majority of cycles")
else:
    print("Conclusion: A notable proportion of cycles exceeded the 400ms threshold")


BAYESIAN ANALYSIS

Test 1: RR-interval recovery (physiological)
Hypothesis: adaptive produces greater RR recovery than static

t(10) = -0.178
BF10 = 0.302
Result: moderate evidence AGAINST the hypothesis

Test 2: SAM arousal change (subjective)
Hypothesis: adaptive produces greater reduction in SAM than static

t(10) = -4.811
BF10 = 53.882
Result: very strong evidence FOR the hypothesis

Test 3: System latency
Checking whether cycle times stayed below 400ms

Total cycles logged: 2625
Mean cycle time: 194.3ms
Max cycle time: 620.1ms
Cycles over 400ms: 12 (0.5%)
Cycles under 400ms: 2613 (99.5%)
Conclusion: Sub-400ms threshold maintained in the vast majority of cycles


## 8. Final results summary

In [9]:
print("SUMMARY")

# Print the introduction
print(f"Bayesian paired t-tests were conducted comparing the adaptive and static conditions across {total_people} participants on two measures.")
print()

# Test 1
print("Physiological measure (RR-interval change):")
print(f"BF10 = {rr_bf10}")
print(f"t({rr_df}) = {rr_t}")
print("Result: " + rr_description)
print()

# Test 2
print("Subjective measure (SAM arousal change):")
print(f"BF10 = {sam_bf10}")
print(f"t({sam_df}) = {sam_t}")
print("Result: " + sam_description)
print()

# Final individual counts
print("Individual level:")
print(f"{number_adaptive_better_rr} out of {total_people} participants showed greater RR recovery under adaptive")
print(f"{number_rr_recovery_adaptive} out of {total_people} showed positive RR change (recovery) under adaptive")
print(f"{number_rr_recovery_static} out of {total_people} showed positive RR change (recovery) under static")
print(f"{number_sam_better_adaptive} out of {total_people} felt calmer (lower SAM) under adaptive")
print(f"{number_sam_better_static} out of {total_people} felt calmer (lower SAM) under static")

SUMMARY
Bayesian paired t-tests were conducted comparing the adaptive and static conditions across 11 participants on two measures.

Physiological measure (RR-interval change):
BF10 = 0.302
t(10) = -0.178
Result: moderate evidence AGAINST the hypothesis

Subjective measure (SAM arousal change):
BF10 = 53.882
t(10) = -4.811
Result: very strong evidence FOR the hypothesis

Individual level:
5 out of 11 participants showed greater RR recovery under adaptive
3 out of 11 showed positive RR change (recovery) under adaptive
2 out of 11 showed positive RR change (recovery) under static
7 out of 11 felt calmer (lower SAM) under adaptive
4 out of 11 felt calmer (lower SAM) under static
